# NB-Step1 · Video Manifest & Inventory
**Pipeline position:** Step 1 of 9 — runs before any annotation remapping or retraining.

### Purpose
Produce a definitive machine-readable record of every `.mp4` available, including:
- Raw video metadata (frames, FPS, resolution, duration, orientation)
- ROI geometry and derived preprocessed-frame dimensions
- Bubble-detection quality assessment via the classical CV pipeline (same as NB-02)
- Coordinate-remap parameters required by **NB-Step3** (annotation alignment)
- Annotation provenance flags

### Outputs (all written to `MANIFEST_DIR`)
| File | Used by |
|------|---------|
| `video_manifest_<ts>.csv` | Human review, pandas downstream |
| `video_manifest_<ts>.json` | Full structured record including remap params |
| `coord_remap_params_<ts>.json` | NB-Step3 (COCO coordinate remapping) |
| `qa_<run_label>.png` | Visual confirmation of preprocessing + detection |

> **Action required in Cell 3:** Set `has_annotations = True` for the video(s) that were
> annotated in CVAT, and fill in the `annot_original_frame_start/end` values.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
import json, os, time, warnings
from pathlib import Path
from datetime import datetime

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from IPython.display import display, HTML
warnings.filterwarnings('ignore')

print("✓ Imports complete")
print(f"  OpenCV  {cv2.__version__}")
print(f"  NumPy   {np.__version__}")
print(f"  Pandas  {pd.__version__}")


In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Drive mounted")


In [ ]:
# ── Cell 3 · Static Configuration ───────────────────────────────────────────
# HUMAN-EDITED ONLY.  No computed values live here.
# Run this cell first; every subsequent cell reads from these objects.

import cv2   # needed for INTER_CUBIC constant

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A"
ANNOT_DIR  = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool/images_29"
MANIFEST_DIR = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/manifests"

# ── Video catalogue ───────────────────────────────────────────────────────────
# Add / uncomment runs as they become available.
# Keys used by later cells: run_label, path, min_bubbles, brightness_thresh.
VIDEO_LIST = [
    {
        "run_label":         "run_001_Q10",
        "path":              f"{DRIVE_BASE}/run_001_Q10/VID_20260413_124627.mp4",
        "frames_to_save":    15,
        "min_bubbles":       1,
        "max_bubbles":       12,
        "brightness_thresh": 2,
        "sample_every_n":    1,
        "min_frame_gap":     3,
    },
    {
        "run_label":         "run_002_Q20",
        "path":              f"{DRIVE_BASE}/run_002_Q20/VID_20260429_124526.mp4",
        "frames_to_save":    115,
        "min_bubbles":       1,
        "max_bubbles":       12,
        "brightness_thresh": 13,
        "sample_every_n":    5,
        "min_frame_gap":     8,
    },
    {
        "run_label":         "run_003_Q30",
        "path":              f"{DRIVE_BASE}/run_003_Q30/VID_20260429_125119.mp4",
        "frames_to_save":    35,
        "min_bubbles":       1,
        "max_bubbles":       12,
        "brightness_thresh": 20,
        "sample_every_n":    2,
        "min_frame_gap":     5,
    },
    # Uncomment when available:
     {"run_label": "run_004_Q40", "path": f"{DRIVE_BASE}/run_004_Q40/VID_20260429_125717.mp4",
      "frames_to_save": 35, "min_bubbles": 1, "max_bubbles": 12,
      "brightness_thresh": 15, "sample_every_n": 3, "min_frame_gap": 8},
     {"run_label": "run_005_Q50", "path": f"{DRIVE_BASE}/run_005_Q50/VID_20260429_123807.mp4",
      "frames_to_save": 35, "min_bubbles": 2, "max_bubbles": 12,
      "brightness_thresh": 20, "sample_every_n": 5, "min_frame_gap": 10},
     {"run_label": "run_006_Q60", "path": f"{DRIVE_BASE}/run_006_Q60/VID_20260504_130057.mp4",
      "frames_to_save": 35, "min_bubbles": 1, "max_bubbles": 15,
      "brightness_thresh": 8,  "sample_every_n": 3, "min_frame_gap": 8},
]

# ── Preprocessing parameters (must match NB-01 exactly) ──────────────────────
PREPROC = {
    "roi":              [0, 450, 2160, 3400],  # [x0, y0, x1, y1] from config.json
    "downscale_factor": 0.5,
    "clahe_clip_limit": 2.0,
    "clahe_tile_grid":  (4, 4),
    "interpolation":    cv2.INTER_CUBIC,
}

# ── COCO annotation file overrides ────────────────────────────────────────────
# Fill in the path to your _annotations.coco.json for each annotated run.
# Leave as None for runs with no annotation file yet.
COCO_FILES = {
    "run_002_Q20": "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool/in_df_147_01.json",
    "run_003_Q30": "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool/in_df_147_01.json",
}

# ── Sampling ──────────────────────────────────────────────────────────────────
N_SAMPLE_FRAMES = 10    # frames sampled per video for QA

# ─────────────────────────────────────────────────────────────────────────────
import os
from datetime import datetime
os.makedirs(MANIFEST_DIR, exist_ok=True)
MANIFEST_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

_roi = PREPROC["roi"]
_ds  = PREPROC["downscale_factor"]
ROI_W_ORIG = _roi[2] - _roi[0]
ROI_H_ORIG = _roi[3] - _roi[1]
PREPROC_W  = int(ROI_W_ORIG * _ds)
PREPROC_H  = int(ROI_H_ORIG * _ds)

print(f"✓ Cell 3 — static configuration loaded")
print(f"  Active runs    : {len(VIDEO_LIST)}")
print(f"  ROI (original) : {_roi}  →  {ROI_W_ORIG}×{ROI_H_ORIG} px")
print(f"  Preprocessed   : {PREPROC_W}×{PREPROC_H} px")
print(f"  Manifest dir   : {MANIFEST_DIR}")


In [ ]:
import json
with open("/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool/harvest_manifest.json") as f:
    hm = json.load(f)

# Structure and first record
print(type(hm))
if isinstance(hm, list):
    print(f"Records: {len(hm)}")
    print("First record keys:", list(hm[0].keys()) if hm else "empty")
    print("First record:", hm[0])
elif isinstance(hm, dict):
    print("Top-level keys:", list(hm.keys()))
    for k, v in hm.items():
        print(f"  {k}: {str(v)[:120]}")

In [ ]:
import json
with open("/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool/harvest_manifest.json") as f:
    hm = json.load(f)

for v in hm["videos"]:
    print(f"\nrun_label : {v['run_label']}")
    print(f"video_path: {v['video_path']}")
    # Print all keys and values except large lists
    for k, val in v.items():
        if k in ("run_label", "video_path"):
            continue
        if isinstance(val, list):
            print(f"{k} ({len(val)} items): {val[:5]} ...")
        else:
            print(f"{k}: {val}")

In [ ]:
# ── Cell 4 · Annotation Discovery ────────────────────────────────────────────
# Reads only directory entries from ANNOT_DIR (zero file content I/O).
# Builds ANNOTATION_CONFIG from filenames alone, using VIDEO_LIST as the
# reverse-lookup table.  No manual pasting required.

import re
from collections import defaultdict

# Reverse lookup: video basename (no extension) → run_label
vid_to_run = {
    os.path.splitext(os.path.basename(cfg["path"]))[0]: cfg["run_label"]
    for cfg in VIDEO_LIST
}

# Filename convention: VID_20260429_124526_frame04165.jpg  (any extension)
PATTERN = re.compile(
    r"(?P<vid>VID_\d{8}_\d{6})_frame(?P<idx>\d+)",
    re.IGNORECASE,
)

if not os.path.isdir(ANNOT_DIR):
    raise FileNotFoundError(
        f"Annotation directory not found:\n  {ANNOT_DIR}\n"
        "Check Drive is mounted and ANNOT_DIR in Cell 3 is correct."
    )

names     = sorted(os.listdir(ANNOT_DIR))
by_run    = defaultdict(list)
unmatched = []

for name in names:
    stem = os.path.splitext(name)[0]
    m    = PATTERN.search(stem)
    if not m:
        unmatched.append(f"{name}  ← pattern mismatch")
        continue
    vid = m.group("vid")
    run = vid_to_run.get(vid)
    if run:
        by_run[run].append(int(m.group("idx")))
    else:
        unmatched.append(f"{name}  ← vid '{vid}' not in VIDEO_LIST")

# ── Build ANNOTATION_CONFIG ───────────────────────────────────────────────────
ANNOTATION_CONFIG = {}
for run, indices in sorted(by_run.items()):
    s = sorted(indices)
    gaps = [s[i+1] - s[i] for i in range(len(s) - 1)] if len(s) > 1 else [0]
    ANNOTATION_CONFIG[run] = {
        "has_annotations":            True,
        "annot_original_frame_start": s[0],
        "annot_original_frame_end":   s[-1],
        "annotation_frame_indices":   s,          # consumed directly by NB-Step2
        "n_annotated_frames":         len(s),
        "n_annotated_instances":      None,        # resolved in Cell 5 from COCO JSON
        "annotation_file":            COCO_FILES.get(run),  # set in Cell 3
        "sampling_min_gap":           min(gaps),
        "sampling_max_gap":           max(gaps),
        "sampling_type":              "contiguous" if max(gaps) == 1 else "sparse",
    }

# Runs in VIDEO_LIST with no annotations
for cfg in VIDEO_LIST:
    if cfg["run_label"] not in ANNOTATION_CONFIG:
        ANNOTATION_CONFIG[cfg["run_label"]] = {
            "has_annotations":            False,
            "annot_original_frame_start": None,
            "annot_original_frame_end":   None,
            "annotation_frame_indices":   [],
            "n_annotated_frames":         0,
            "n_annotated_instances":      None,
            "annotation_file":            None,
            "sampling_min_gap":           None,
            "sampling_max_gap":           None,
            "sampling_type":              None,
        }

# ── Report ────────────────────────────────────────────────────────────────────
W = 66
print("=" * W)
print("  Cell 4 — Annotation Discovery".center(W))
print("=" * W)
print(f"  Images in pool  : {len(names)}")
print(f"  Source runs     : {len(by_run)}")
print(f"  Unmatched files : {len(unmatched)}")
print()

for run, ac in sorted(ANNOTATION_CONFIG.items()):
    marker = "★ ANNOTATED" if ac["has_annotations"] else "  not annotated"
    print(f"  {run}  —  {marker}")
    if ac["has_annotations"]:
        s = ac["annotation_frame_indices"]
        print(f"    Frames           : {ac['n_annotated_frames']}")
        print(f"    Index range      : {s[0]}  →  {s[-1]}")
        print(f"    Span (raw)       : {s[-1] - s[0] + 1} frames")
        print(f"    Sampling         : {ac['sampling_type']}  "
              f"(gap min={ac['sampling_min_gap']}  max={ac['sampling_max_gap']})")
        print(f"    COCO file        : {ac['annotation_file'] or '⚠  not set — fill COCO_FILES in Cell 3'}")
    print()

if unmatched:
    print(f"  ⚠  Unmatched entries (first 5):")
    for u in unmatched[:5]:
        print(f"     {u}")
    print()

print("✓ ANNOTATION_CONFIG built — proceeding to Cell 5")


In [ ]:
# ── Cell 5 · Merge & Validate → PIPELINE_CONFIG ──────────────────────────────
# Joins VIDEO_LIST (Cell 3) + ANNOTATION_CONFIG (Cell 4) into a single
# PIPELINE_CONFIG dict.  All downstream cells consume only PIPELINE_CONFIG.
# Also resolves n_annotated_instances from the COCO JSON if the file exists.

import json as _json

def _count_coco_instances(coco_path):
    """Return annotation count from a COCO JSON without loading images."""
    if not coco_path or not os.path.exists(coco_path):
        return None
    with open(coco_path) as f:
        data = _json.load(f)
    return len(data.get("annotations", []))

# ── Build PIPELINE_CONFIG ─────────────────────────────────────────────────────
PIPELINE_CONFIG = {
    "manifest_ts":   MANIFEST_TS,
    "preproc":       PREPROC,
    "preproc_w":     PREPROC_W,
    "preproc_h":     PREPROC_H,
    "manifest_dir":  MANIFEST_DIR,
    "annot_dir":     ANNOT_DIR,
    "runs":          {},
}

_vid_cfg = {cfg["run_label"]: cfg for cfg in VIDEO_LIST}

for run_label, ann in ANNOTATION_CONFIG.items():
    vid_cfg = _vid_cfg.get(run_label, {})

    # Resolve instance count from COCO file if available
    n_inst = _count_coco_instances(ann.get("annotation_file"))
    if n_inst is not None:
        ann["n_annotated_instances"] = n_inst

    # Coordinate remap parameters — consumed verbatim by NB-Step3
    roi = PREPROC["roi"]
    ds  = PREPROC["downscale_factor"]
    remap = {
        "roi_x0":    roi[0],  "roi_y0":  roi[1],
        "roi_x1":    roi[2],  "roi_y1":  roi[3],
        "offset_x":  roi[0],  "offset_y": roi[1],
        "scale_x":   ds,      "scale_y":  ds,
        "preproc_w": PREPROC_W,
        "preproc_h": PREPROC_H,
        "formula": {
            "x_p": "(x_orig - offset_x) * scale_x",
            "y_p": "(y_orig - offset_y) * scale_y",
            "w_p": "w_orig * scale_x",
            "h_p": "h_orig * scale_y",
        },
    }

    PIPELINE_CONFIG["runs"][run_label] = {
        "run_label":      run_label,
        "path":           vid_cfg.get("path"),
        "harvest_params": {k: v for k, v in vid_cfg.items()
                           if k not in ("run_label", "path")},
        "annotation":     ann,
        "coord_remap":    remap,
    }

# ── Validation ────────────────────────────────────────────────────────────────
W = 66
issues = []

for run_label, rc in PIPELINE_CONFIG["runs"].items():
    path = rc["path"]
    if not path:
        issues.append(f"{run_label}: no path defined")
        continue
    if not os.path.exists(path):
        issues.append(f"{run_label}: file not found  →  {path}")
        continue
    ann = rc["annotation"]
    if ann["has_annotations"]:
        if not ann["annotation_file"]:
            issues.append(f"{run_label}: ⚠  annotation_file not set in COCO_FILES (Cell 3)")
        if ann["n_annotated_instances"] is None:
            issues.append(f"{run_label}: ⚠  n_annotated_instances unknown (set annotation_file)")

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * W)
print("  Cell 5 — PIPELINE_CONFIG".center(W))
print("=" * W)
print(f"  Runs merged     : {len(PIPELINE_CONFIG['runs'])}")
print(f"  Validation      : {'PASSED' if not issues else f'{len(issues)} warnings'}")
print()

hdr = f"  {'Run label':<20} {'Annotated':^11} {'Frames':>7} {'Instances':>10}  COCO file"
print(hdr)
print(f"  {'─'*20} {'─'*11} {'─'*7} {'─'*10}  {'─'*20}")

for run_label, rc in sorted(PIPELINE_CONFIG["runs"].items()):
    ann = rc["annotation"]
    has  = "★ YES" if ann["has_annotations"] else "no"
    nf   = ann["n_annotated_frames"]   or "—"
    ni   = ann["n_annotated_instances"] or "?"
    cf   = os.path.basename(ann["annotation_file"]) if ann["annotation_file"] else "not set"
    print(f"  {run_label:<20} {has:^11} {str(nf):>7} {str(ni):>10}  {cf}")

if issues:
    print()
    print("  Warnings:")
    for iss in issues:
        print(f"    ⚠  {iss}")

print()
print("✓ PIPELINE_CONFIG ready — all downstream cells use this object.")


In [ ]:
# ── Cell 6 · Helper Functions ─────────────────────────────────────────────────
# All functions used by the inventory loop (Cell 7) and QA panels (Cell 8).
# Sourced from PIPELINE_CONFIG — no direct references to Cell 3 variables.

def get_video_metadata(path: str) -> dict | None:
    """Read video properties via OpenCV. Returns None if file cannot be opened."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return None
    fps      = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {
        "total_frames": n_frames,
        "fps":          round(fps, 6),
        "width_px":     width,
        "height_px":    height,
        "duration_s":   round(n_frames / fps, 3) if fps > 0 else None,
        "orientation":  "portrait" if height > width else "landscape",
    }

def build_remap_params(roi: list, ds: float) -> dict:
    x0, y0, x1, y1 = roi
    return {
        "roi_x0":    x0,  "roi_y0":  y0,
        "roi_x1":    x1,  "roi_y1":  y1,
        "offset_x":  x0,  "offset_y": y0,
        "scale_x":   ds,  "scale_y":  ds,
        "preproc_w": int((x1 - x0) * ds),
        "preproc_h": int((y1 - y0) * ds),
    }


def validate_roi(frame_w: int, frame_h: int, roi: list) -> list:
    """Return list of ROI geometry problems (empty list = OK)."""
    x0, y0, x1, y1 = roi
    issues = []
    if x0 < 0 or y0 < 0:
        issues.append(f"ROI origin is negative ({x0},{y0})")
    if x1 > frame_w:
        issues.append(f"x1={x1} exceeds frame width={frame_w}")
    if y1 > frame_h:
        issues.append(f"y1={y1} exceeds frame height={frame_h}")
    if x1 <= x0:
        issues.append(f"x1={x1} <= x0={x0}")
    if y1 <= y0:
        issues.append(f"y1={y1} <= y0={y0}")
    return issues


def preprocess_frame(frame_bgr: np.ndarray, preproc: dict) -> np.ndarray:
    """Apply the NB-01 chain: ROI crop → grayscale → 0.5x bicubic → CLAHE."""
    x0, y0, x1, y1 = preproc["roi"]
    h, w  = frame_bgr.shape[:2]
    crop  = frame_bgr[max(0,y0):min(h,y1), max(0,x0):min(w,x1)]
    gray  = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop.copy()
    nw    = int(gray.shape[1] * preproc["downscale_factor"])
    nh    = int(gray.shape[0] * preproc["downscale_factor"])
    small = cv2.resize(gray, (nw, nh), interpolation=preproc["interpolation"])
    clahe = cv2.createCLAHE(
        clipLimit    = preproc["clahe_clip_limit"],
        tileGridSize = preproc["clahe_tile_grid"],
    )
    return clahe.apply(small)


def detect_bubbles_classical(proc: np.ndarray) -> tuple:
    """
    Classical CV bubble detector matching NB-02.
    Returns (count, binary_mask, centroids_xy).
    """
    blurred       = cv2.GaussianBlur(proc, (5, 5), 0)
    _, otsu       = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    p92           = float(np.percentile(blurred, 92))
    _, p92m       = cv2.threshold(blurred, p92, 255, cv2.THRESH_BINARY)
    combined      = cv2.bitwise_or(otsu, p92m)
    kernel        = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask          = cv2.morphologyEx(combined, cv2.MORPH_OPEN,  kernel)
    mask          = cv2.morphologyEx(mask,     cv2.MORPH_CLOSE, kernel)
    n_labels, _, stats, centroids = cv2.connectedComponentsWithStats(mask)
    valid_idx     = [i for i in range(1, n_labels)
                     if 10 <= stats[i, cv2.CC_STAT_AREA] <= 3000]
    cents         = centroids[valid_idx] if valid_idx else np.zeros((0, 2))
    return len(valid_idx), mask, cents


def compute_snr_db(proc: np.ndarray) -> float:
    """Rough SNR estimate: bright-pixel mean over dark-pixel std."""
    med   = np.median(proc)
    sig   = proc[proc >  med]
    noise = proc[proc <= med]
    if len(sig) < 5 or noise.std() < 0.5:
        return 99.0
    return float(20 * np.log10(sig.mean() / noise.std()))


def sample_video(path: str, n: int, preproc: dict) -> tuple:
    """
    Sample n evenly-spaced frames, apply preprocessing, run classical detector.
    Returns (per_frame_stats, qa_frames).
      per_frame_stats : list of dicts  {frame_idx, n_bubbles, snr_db, brightness}
      qa_frames       : list of tuples (frame_idx, proc, mask, centroids)
    """
    cap     = cv2.VideoCapture(path)
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, n, dtype=int)
    stats, qa = [], []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        proc             = preprocess_frame(frame, preproc)
        n_bub, mask, cen = detect_bubbles_classical(proc)
        stats.append({
            "frame_idx":  int(idx),
            "n_bubbles":  n_bub,
            "snr_db":     round(compute_snr_db(proc), 2),
            "brightness": round(float(proc.mean()), 2),
        })
        qa.append((int(idx), proc, mask, cen))
    cap.release()
    return stats, qa


print("✓ Cell 6 — helper functions defined")
print(f"  get_video_metadata, validate_roi, preprocess_frame,")
print(f"  detect_bubbles_classical, compute_snr_db, sample_video")


In [ ]:
# ── Main inventory loop ────────────────────────────────────────────────────────

manifest_rows  = []   # one dict per video → DataFrame
coord_remap    = {}   # run_label → remap params  (written to coord_remap_params.json)
qa_store       = {}   # run_label → qa_frames     (used in Cell 6)

for cfg in VIDEO_LIST:
    label = cfg["run_label"]
    path  = cfg["path"]

    print(f"\n{'─'*64}")
    print(f"  {label}")
    print(f"  {path}")

    row = {
        "run_label":    label,
        "path":         path,
        "file_exists":  os.path.exists(path),
        "file_size_mb": round(os.path.getsize(path) / 1e6, 1)
                        if os.path.exists(path) else None,
    }

    # ── Missing file ─────────────────────────────────────────────────────────
    if not row["file_exists"]:
        print("  ✗  FILE NOT FOUND — skipping QA")
        row.update({
            "total_frames": None, "fps": None, "duration_s": None,
            "width_px": None, "height_px": None, "orientation": None,
            "roi_valid": None, "roi_issues": "file missing",
            "preproc_w": None, "preproc_h": None,
            "bubble_mean": None, "bubble_std": None,
            "bubble_min": None, "bubble_max": None,
            "snr_mean_db": None, "brightness_mean": None,
            "quality_flag": "FILE_MISSING",
            "recommended_use": "EXCLUDE",
            "has_annotations": False,
            "annot_frame_start": None, "annot_frame_end": None,
            "n_annot_instances": None, "n_annot_frames": None,
            "annotation_file": None, "annotation_notes": "",
        })
        manifest_rows.append(row)
        continue

    # ── Video metadata ────────────────────────────────────────────────────────
    meta = get_video_metadata(path)
    if meta is None:
        print("  ✗  Cannot open video with OpenCV")
        row["quality_flag"]    = "UNREADABLE"
        row["recommended_use"] = "EXCLUDE"
        manifest_rows.append(row)
        continue

    row.update(meta)
    print(f"  {meta['total_frames']:,} frames  |  {meta['fps']:.3f} fps  |  "
          f"{meta['width_px']}×{meta['height_px']}  |  "
          f"{meta['duration_s']:.1f} s  |  {meta['orientation'].upper()}")

    # ── ROI validation ────────────────────────────────────────────────────────
    roi_issues = validate_roi(meta["width_px"], meta["height_px"], PREPROC["roi"])
    row["roi_valid"]  = len(roi_issues) == 0
    row["roi_issues"] = "; ".join(roi_issues) if roi_issues else ""
    if roi_issues:
        print(f"  ⚠  ROI issues: {roi_issues}")

    # ── Derived geometry ──────────────────────────────────────────────────────
    rp = build_remap_params(PREPROC["roi"], PREPROC["downscale_factor"])
    row["preproc_w"] = rp["preproc_w"]
    row["preproc_h"] = rp["preproc_h"]
    coord_remap[label] = rp
    print(f"  Preprocessed frame : {rp['preproc_w']}×{rp['preproc_h']} px")

    # ── Frame sampling & QA ───────────────────────────────────────────────────
    print(f"  Sampling {N_SAMPLE_FRAMES} frames ...", end=" ", flush=True)
    t0 = time.time()
    per_frame, qa_frames = sample_video(path, N_SAMPLE_FRAMES, PREPROC)
    print(f"done ({time.time()-t0:.1f}s)")

    if per_frame:
        bc = [r["n_bubbles"] for r in per_frame]
        sn = [r["snr_db"]    for r in per_frame]
        br = [r["brightness"] for r in per_frame]
        row.update({
            "bubble_mean":     round(float(np.mean(bc)),  2),
            "bubble_std":      round(float(np.std(bc)),   2),
            "bubble_min":      int(np.min(bc)),
            "bubble_max":      int(np.max(bc)),
            "snr_mean_db":     round(float(np.mean(sn)),  2),
            "brightness_mean": round(float(np.mean(br)),  2),
        })
        print(f"  Bubble det  mean={row['bubble_mean']:.1f}  "
              f"range=[{row['bubble_min']}–{row['bubble_max']}]  "
              f"SNR={row['snr_mean_db']:.1f} dB  "
              f"brightness={row['brightness_mean']:.1f}")

        # Quality flag
        if   not row["roi_valid"]:              qflag = "ROI_MISMATCH"
        elif row["snr_mean_db"]  < 5.0:        qflag = "LOW_SNR"
        elif row["bubble_mean"]  < 0.5:        qflag = "NO_BUBBLES_DETECTED"
        elif row["bubble_mean"]  < 1.0:        qflag = "SPARSE_BUBBLES"
        else:                                  qflag = "OK"
        row["quality_flag"] = qflag

        # Recommended use
        if   qflag == "OK" and row["bubble_mean"] >= cfg.get("min_bubbles", 1):
            row["recommended_use"] = "TRAINING_AND_INFERENCE"
        elif qflag in ("SPARSE_BUBBLES", "LOW_SNR"):
            row["recommended_use"] = "INFERENCE_ONLY"
        elif qflag == "OK":
            row["recommended_use"] = "TRAINING_AND_INFERENCE"
        else:
            row["recommended_use"] = "CHECK_REQUIRED"

        qa_store[label] = qa_frames

    # ── Annotation provenance ─────────────────────────────────────────────────
    ac = ANNOTATION_CONFIG.get(label, {})
    row.update({
        "has_annotations":   ac.get("has_annotations",             False),
        "annot_frame_start": ac.get("annot_original_frame_start",  None),
        "annot_frame_end":   ac.get("annot_original_frame_end",    None),
        "n_annot_instances": ac.get("n_annotated_instances",       None),
        "n_annot_frames":    ac.get("n_annotated_frames",          None),
        "annotation_file":   ac.get("annotation_file",             None),
        "annotation_notes":  ac.get("notes",                       ""),
    })

    if row["has_annotations"]:
        fs  = row["annot_frame_start"] or "?"
        fe  = row["annot_frame_end"]   or "?"
        ni  = row["n_annot_instances"] or "?"
        nf  = row["n_annot_frames"]    or "?"
        print(f"  ★  ANNOTATED  frames {fs}–{fe}  |  {ni} instances  |  {nf} frames")

    manifest_rows.append(row)
    print(f"  → {row['quality_flag']}  /  {row['recommended_use']}")

print(f"\n{'='*64}")
print(f"✓ Inventory complete — {len(manifest_rows)} videos processed")


In [ ]:
# ── Visual QA panels ──────────────────────────────────────────────────────────
# Each panel shows N sample preprocessed frames with detected bubble centroids
# overlaid in green.  Frames with zero detections are titled in red.

def render_qa_panel(label: str, qa_frames: list, n_cols: int = 5) -> str:
    """Render and save a QA panel; return the saved path."""
    n      = min(len(qa_frames), 10)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.8, n_rows * 4.2))
    axes = np.array(axes).flatten()

    for i, (fidx, proc, mask, cents) in enumerate(qa_frames[:n]):
        ax = axes[i]
        ax.imshow(proc, cmap="gray", aspect="auto")
        if len(cents) > 0:
            ax.scatter(cents[:, 0], cents[:, 1],
                       c="lime", s=18, marker="+", linewidths=0.9, zorder=3)
        color = "lime" if len(cents) >= 1 else "tomato"
        ax.set_title(f"f={fidx}\n{len(cents)} det", fontsize=7.5, color=color, pad=2)
        ax.axis("off")

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"{label}  —  QA sample ({n} frames, green = detections)",
                 fontsize=10, fontweight="bold", y=1.01)
    plt.tight_layout(pad=0.4)
    out = os.path.join(MANIFEST_DIR, f"qa_{label}.png")
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.close(fig)
    return out


print("Rendering QA panels …")
for label, qa_frames in qa_store.items():
    out = render_qa_panel(label, qa_frames)
    print(f"  ✓  {label}  →  {out}")
print("Done.")


In [ ]:
# ── Export manifest files ──────────────────────────────────────────────────────

# ── 1. DataFrame & CSV ────────────────────────────────────────────────────────
COL_ORDER = [
    "run_label", "file_exists", "file_size_mb",
    "total_frames", "fps", "duration_s",
    "width_px", "height_px", "orientation",
    "roi_valid", "roi_issues",
    "preproc_w", "preproc_h",
    "bubble_mean", "bubble_std", "bubble_min", "bubble_max",
    "snr_mean_db", "brightness_mean",
    "quality_flag", "recommended_use",
    "has_annotations", "annot_frame_start", "annot_frame_end",
    "n_annot_instances", "n_annot_frames",
    "annotation_file", "annotation_notes",
    "path",
]
df = pd.DataFrame(manifest_rows)
df = df[[c for c in COL_ORDER if c in df.columns]]

csv_path = os.path.join(MANIFEST_DIR, f"video_manifest_{MANIFEST_TS}.csv")
df.to_csv(csv_path, index=False)
print(f"✓ CSV  →  {csv_path}")

# ── 2. Full JSON manifest ─────────────────────────────────────────────────────
def to_serialisable(v):
    if isinstance(v, (np.integer,)):  return int(v)
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, np.ndarray):     return v.tolist()
    return v

preproc_serial = {k: (list(v) if isinstance(v, tuple) else v)
                  for k, v in PREPROC.items()
                  if k != "interpolation"}   # cv2 flag not JSON-serialisable

manifest_json = {
    "schema_version":  "1.0",
    "generated_at":    MANIFEST_TS,
    "preprocessing":   preproc_serial,
    "n_videos":        len(manifest_rows),
    "videos": [
        {
            **{k: to_serialisable(v) for k, v in row.items()},
            "coord_remap_params": coord_remap.get(row["run_label"]),
        }
        for row in manifest_rows
    ],
}

json_path = os.path.join(MANIFEST_DIR, f"video_manifest_{MANIFEST_TS}.json")
with open(json_path, "w") as fh:
    json.dump(manifest_json, fh, indent=2, default=str)
print(f"✓ JSON →  {json_path}")

# ── 3. Coord-remap params (standalone — consumed directly by NB-Step3) ────────
remap_out = {
    "schema_version": "1.0",
    "generated_at":   MANIFEST_TS,
    "purpose": (
        "Remap COCO annotation coordinates from original video space "
        "to preprocessed-frame space.  Consumed by NB-Step3."
    ),
    "global_formula": {
        "x_preproc":  "(x_orig - offset_x) * scale_x",
        "y_preproc":  "(y_orig - offset_y) * scale_y",
        "bbox_w":     "bbox_w_orig * scale_x",
        "bbox_h":     "bbox_h_orig * scale_y",
        "note":       "Apply to EVERY point in bbox [x,y,w,h] AND segmentation polygon.",
    },
    "per_video": coord_remap,
}

remap_path = os.path.join(MANIFEST_DIR, f"coord_remap_params_{MANIFEST_TS}.json")
with open(remap_path, "w") as fh:
    json.dump(remap_out, fh, indent=2)
print(f"✓ Remap →  {remap_path}")


In [ ]:
# ── Summary report ────────────────────────────────────────────────────────────

W = 72
print("=" * W)
print("  NB-Step1 · VIDEO MANIFEST — SUMMARY".center(W))
print(f"  {MANIFEST_TS}".center(W))
print("=" * W)

total   = len(df)
found   = int(df["file_exists"].sum())
ok      = int((df["quality_flag"] == "OK").sum())
annot   = int(df["has_annotations"].sum())
unconf  = int((df["has_annotations"] == False).sum())

print(f"\n  Videos configured  : {total}")
print(f"  Files found        : {found} / {total}")
print(f"  Quality OK         : {ok} / {found}")
print(f"  Confirmed annotated: {annot}")
if annot == 0:
    print("  ⚠  No annotated video confirmed — update ANNOTATION_CONFIG in Cell 3.")

print()
hdr = (f"  {'Run label':<20} {'Frames':>7} {'FPS':>8} "
       f"{'Det/fr':>7} {'SNR dB':>7} {'Quality':<22} Use")
print(hdr)
print(f"  {'─'*20} {'─'*7} {'─'*8} {'─'*7} {'─'*7} {'─'*22} {'─'*24}")

for _, r in df.iterrows():
    if not r["file_exists"]:
        print(f"  {r['run_label']:<20}  ← FILE NOT FOUND")
        continue
    star = " ★" if r.get("has_annotations") else ""
    print(
        f"  {r['run_label']:<20} "
        f"{r['total_frames']:>7,.0f} "
        f"{r['fps']:>8.3f} "
        f"{r['bubble_mean']:>7.1f} "
        f"{r['snr_mean_db']:>7.1f} "
        f"{r['quality_flag']:<22}"
        f"{r['recommended_use']}{star}"
    )

# Geometry block
print(f"\n  Preprocessed frame dimensions (all videos share ROI + scale):")
print(f"    Original ROI  : {PREPROC['roi']}  →  {ROI_W_ORIG}×{ROI_H_ORIG} px")
print(f"    After 0.5×    : {PREPROC_W}×{PREPROC_H} px")

# Annotation block
ann_df = df[df["has_annotations"] == True]
if len(ann_df) > 0:
    print("\n  ★  ANNOTATED VIDEOS — coordinate remap parameters:")
    for _, r in ann_df.iterrows():
        rp = coord_remap.get(r["run_label"], {})
        fs = r["annot_frame_start"] or "?"
        fe = r["annot_frame_end"]   or "?"
        print(f"    {r['run_label']}")
        print(f"      Annotated frames  : {fs} – {fe}")
        print(f"      Instances / frames: "
              f"{r['n_annot_instances'] or '?'} / {r['n_annot_frames'] or '?'}")
        if rp:
            print(f"      x_p = (x_orig − {rp['offset_x']}) × {rp['scale_x']}")
            print(f"      y_p = (y_orig − {rp['offset_y']}) × {rp['scale_y']}")
            print(f"      Valid preproc range: [0,{rp['preproc_w']}] × [0,{rp['preproc_h']}]")

print(f"\n  Output files")
print(f"    CSV   : {csv_path}")
print(f"    JSON  : {json_path}")
print(f"    Remap : {remap_path}")
print(f"    QA    : {MANIFEST_DIR}/qa_<run_label>.png")
print("=" * W)
print()
print("  ✓ Step 1 complete.")
print("  Next → NB-Step2: re-extract annotated frames through NB-01 pipeline.")
print("          Input  : annotated source mp4  (run confirmed above)")
print("          Key    : use coord_remap_params_<ts>.json for NB-Step3.")
